In [ ]:
# now need to calibrate our rgb to wavelength conversion
# going to set calibration as helium bcos it has the widest range

helium_bands = { # pixel pos : real wavelength
    825: 388.8,
    1285: 447.1,
    1435: 471.3,
    1566: 492.1,
    1612: 501.5,
    2136: 587.5,
    2627: 667.8
}

#print(picFiles_list)
he_rgb = get_rgb_nobackground('pics/g6f_he_2_full.dng') # helium
display(he_rgb)

calibration = he_rgb[he_rgb["pixel"].isin(helium_bands) 
                     & he_rgb["valid"]].copy()
#print(calibration)
calibration["wavelength_nm"] = calibration["pixel"].map(helium_bands)

# gonna fit linear
# im pretty sure the camera response is NOT linear
# but we only have 6 data points from helium and i dont want to use the rest
# bcos iw to validate with them

X = calibration[["r", "g", "b"]].to_numpy()
y = calibration["wavelength_nm"].to_numpy()

model = Ridge(alpha = 0.001) 
# alpha does... something... to control regulation strength
# idk what but docs says it must be non negative number. okay
model.fit(X, y)
calibration["predicted_nm"] = model.predict(X)

display(calibration[["pixel", "wavelength_nm", "predicted_nm"]])


In [ ]:
# lets go see the rest

def test_element(pic, known_lines=None):
    # Extract RGB measurements using existing function
    data = get_rgb_nobackground(pic)

    # Exclude invalid RGB measurements
    data = data[data["valid"]].copy()

    # Predict wavelengths using the helium model
    X = data[["r", "g", "b"]].to_numpy()

    data["predicted_nm"] = model.predict(X)

    # put ur known lines
    if known_lines is not None:
        data["actual_nm"] = data["pixel"].map(known_lines)
        data["error_nm"] = data["predicted_nm"] - data["actual_nm"]

    # display(data[["pixel", "predicted_nm", "actual_nm", "error_nm"]].round(2))
    display(data)

    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(
        data["pixel"],
        data["predicted_nm"],
        s=60
    )

    for _, row in data.iterrows():
        ax.annotate(
            f'{row["predicted_nm"]:.1f}',
            (row["pixel"], row["predicted_nm"]),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=8
        )

    ax.set_xlabel("Detected peak position (pixels)")
    ax.set_ylabel("Predicted wavelength (nm)")
    ax.set_title(pic)
    ax.grid(alpha=0.3)

    plt.show()

    if known_lines is not None:
        matched = data.dropna(subset=["actual_nm"])

        if len(matched):
            mae = np.mean(np.abs(matched["error_nm"]))
            print(f"Validation MAE: {mae:.2f} nm")

    return data

In [ ]:
h_file = "pics/g6f_h_full.dng"

# pixel pos:legit
hydrogen_lines = {
    1732: 486,
    2666: 656 
}

h_results = test_element(
    h_file,
    known_lines=hydrogen_lines
)

In [ ]:
hg_file = "pics/g6f_hg_full.dng"

# pos:legit
mercury_lines = {
    1344: 404.7,
    1501: 435.8,
    2081: 546.1,
    2255: 576.9 # there's another at 579 which we can't resolve
}

hg_results = test_element(
    hg_file,
    known_lines=mercury_lines
)

In [ ]:
# this deuterium is highkey suspicious
# is it neon?????
# time to check

def unknown_ele(file):
    rgb = get_rgb_nobackground(file)

    valid = rgb[rgb["valid"]].copy()

    # Use the same features as your helium calibration
    sus = valid[["r", "g", "b"]].to_numpy()

    valid["estimated_nm"] = model.predict(sus)

    return valid

In [ ]:
sus_deu = unknown_ele("pics/g6f_deu_full.dng")

display(sus_deu[["pixel", "estimated_nm"]])

In [ ]:

datasets = {
    "Helium": (he, helium_bands),
    "Hydrogen": (h, hydrogen_lines),
    "Mercury": (hg, mercury_lines)
}

for name, (data, references) in datasets.items():

    test = data.copy()
    test["actual_nm"] = test["pixel"].map(references)

    test = test.dropna(subset=["actual_nm"])
    test = test[
        test["actual_nm"].between(400, 700)
    ].copy()

    test["original_nm"], _ = predict_sensor_wavelength(test)
    test["corrected_nm"], _ = predict_corrected_wavelength(test)

    original_mae = np.mean(
        np.abs(test["original_nm"] - test["actual_nm"])
    )

    corrected_mae = np.mean(
        np.abs(test["corrected_nm"] - test["actual_nm"])
    )

    print(f"\n{name}")
    display(test[
        ["pixel", "actual_nm", "original_nm", "corrected_nm"]
    ].round(2))

    print(f"Original MAE:  {original_mae:.2f} nm")
    print(f"Corrected MAE: {corrected_mae:.2f} nm")


In [ ]:
# testing helium calibration code below

In [ ]:
from scipy.optimize import least_squares
from scipy.interpolate import PchipInterpolator
import numpy as np

# Use only known, valid helium lines within sensor range
he_cal = he[he["pixel"].isin(helium_bands)].copy()
he_cal["actual_nm"] = he_cal["pixel"].map(helium_bands)

he_cal = he_cal[
    he_cal["actual_nm"].between(400, 700)
].dropna(subset=["r", "g", "b"])

wl = sensor["wavelength_nm"].to_numpy()
responses = sensor[["R", "G", "B"]].to_numpy()

curves = [
    PchipInterpolator(wl, responses[:, i])
    for i in range(3)
]

known_wl = he_cal["actual_nm"].to_numpy()
measured_rgb = he_cal[["r", "g", "b"]].to_numpy()

# Published sensor response at known helium wavelengths
theoretical_rgb = np.column_stack([
    curve(known_wl) for curve in curves
])

def residuals(log_gains):
    # Exponentiation ensures gains remain positive
    kR, kB = np.exp(log_gains)

    corrected = theoretical_rgb * [kR, 1.0, kB]
    corrected /= corrected.sum(axis=1, keepdims=True)

    return (corrected - measured_rgb).ravel()

fit = least_squares(
    residuals,
    x0=[0, 0],
    bounds=([-2, -2], [2, 2])
)

kR, kB = np.exp(fit.x)

print(f"Red correction factor: {kR:.4f}")
print(f"Green correction factor: 1.0000")
print(f"Blue correction factor: {kB:.4f}")
print("Optimization successful:", fit.success)


In [ ]:

# Original interpolated response curves
corrected_rgb_grid = np.column_stack([
    curve(wl_grid) for curve in curves
])

# Apply channel corrections
corrected_rgb_grid *= [kR, 1.0, kB]

# Normalize
corrected_rgb_grid = np.clip(corrected_rgb_grid, 0, None)
corrected_rgb_grid /= corrected_rgb_grid.sum(
    axis=1, keepdims=True
)

def predict_corrected_wavelength(data):
    measured = data[["r", "g", "b"]].to_numpy()

    distances = np.linalg.norm(
        measured[:, None, :] -
        corrected_rgb_grid[None, :, :],
        axis=2
    )

    nearest = distances.argmin(axis=1)

    return (
        wl_grid[nearest],
        distances[np.arange(len(nearest)), nearest]
    )


In [ ]:
# old background window method
def get_bayer_rgb(region): # old, not needed for current bg method
    '''average bayer rgb in the region'''
    if region.size == 0:
        return None

    # identifying bayer colours ughhh
    B = region[0::2, 0::2].mean()
    G1 = region[0::2, 1::2].mean()
    G2 = region[1::2, 0::2].mean()
    R = region[1::2, 1::2].mean()

    return np.array([R, (G1+G2)/2, B]) # RGB

def get_band_nobackground(img, peak_x, y0, y1, half_width = 8, 
                          bg_gap = 15, bg_width = 12, subtract_bg = True,
                          occupied = None, max_search = 120):
    h, w = img.shape

    # row stuff
    def even(x):
        return 2 * (int(x) // 2)

    # Ensure rows are aligned to Bayer blocks
    y0 = max(0, 2 * (y0 // 2))
    y1 = min(h, 2 * (y1 // 2))

    x0 = max(0, even(peak_x - half_width))
    x1 = min(w, even(peak_x + half_width + 2))

    signal = get_bayer_rgb(img[y0:y1, x0:x1])

    if not subtract_bg:
        return signal

    if occupied is None:
        occupied = []

    def is_clean(a, b):
        if a < 0 or b > w or b <= a:
            return False

        return all(
            b <= left or a >= right
            for left, right in occupied
        )

    def find_background(side):
        # Search outward in even-pixel increments
        for extra in range(0, max_search + 1, 2):
            if side == "left":
                b = even(x0 - bg_gap - extra)
                a = even(b - bg_width)
            else:
                a = even(x1 + bg_gap + extra)
                b = even(a + bg_width)

            if is_clean(a, b):
                return get_bayer_rgb(img[y0:y1, a:b])

        return None

    left = find_background("left")
    right = find_background("right")

    backgrounds = [
        bg for bg in (left, right)
        if bg is not None
    ]

    if not backgrounds:
        return np.full(3, np.nan)

    background = np.mean(backgrounds, axis=0)

    return signal - background

def get_rgb_nobackground(pic):
    img = get_full_pic(pic)
    start, end = get_band(pic)

    sum_lst = vert_sum(pic)
    smoothed, peaks = find_peaks_smoothed(sum_lst)

    widths, heights, left_ips, right_ips = peak_widths(
    smoothed, peaks, rel_height=0.5
    )

    # Occupied regions include a safety margin
    margin = 8

    occupied = [
        (int(np.floor(left - margin)),
         int(np.ceil(right + margin)))
        for left, right in zip(left_ips, right_ips)
    ]

    results = []

    for x, width in zip(peaks, widths):
        half_width = max(4, int(np.ceil(width / 2)))

        rgb = get_band_nobackground(
            img, int(x), start, end,
            half_width=half_width,
            occupied = occupied
        )

        valid = (np.all(np.isfinite(rgb)) and np.all(rgb > 0))

        if valid:
            normalised = rgb / rgb.sum()
        else:
            normalised = [np.nan, np.nan, np.nan]
            print(f"Invalid peak: {x}")

        results.append({
        "pixel": int(x),
        "R": rgb[0],
        "G": rgb[1],
        "B": rgb[2],
        "r": normalised[0],
        "g": normalised[1],
        "b": normalised[2],
        "valid": valid})

    return pd.DataFrame(results)
        
